In [16]:
import pandas as pd

df = pd.read_csv("data/airbnb.csv")
df = df.dropna()
df = df.sample(5000, random_state=42)
print(df.head())

       price_total        room_type  is_shared_room  is_private_room  \
12395    80.954548     Private room               0                1   
44703   166.628100  Entire home/apt               0                0   
49956  1166.837733  Entire home/apt               0                0   
6129    150.529772  Entire home/apt               0                0   
3227    306.221251  Entire home/apt               0                0   

       max_guests  is_superhost  is_multi_listing  is_business_listing  \
12395         2.0             1                 0                    0   
44703         4.0             1                 0                    0   
49956         6.0             1                 0                    1   
6129          5.0             1                 0                    0   
3227          4.0             0                 1                    0   

       cleanliness_score  guest_satisfaction_score  ...  Safety_Index  \
12395               10.0                     100.

Exploratory analysis:


In [17]:
# remove extreme outliers
upper = df['price_total'].quantile(0.99)
df = df[df['price_total'] < upper]
df = df.reset_index(drop=True)

In [18]:
print(df.shape)

(4950, 36)


In [19]:
df = df.drop_duplicates()

In [20]:
# check null vals
print(df.isnull().sum())

price_total                       0
room_type                         0
is_shared_room                    0
is_private_room                   0
max_guests                        0
is_superhost                      0
is_multi_listing                  0
is_business_listing               0
cleanliness_score                 0
guest_satisfaction_score          0
num_bedrooms                      0
distance_city_center              0
distance_metro                    0
attraction_index                  0
attraction_index_norm             0
restaurant_index                  0
restaurant_index_norm             0
longitude                         0
latitude                          0
city                              0
day_type                          0
district                          0
state                             0
country_code                      0
country_name                      0
Crime_Index                       0
Safety_Index                      0
Monthly_Average_Net_salary  

Amelia Viz

Visualization 1

In [21]:
city_order = df.groupby('city')['price_total'].median().sort_values(ascending = False).index.tolist()

chart = alt.Chart(df).mark_boxplot().encode(x = alt.X('price_total:Q', title = 'Nightly Price (€)'),
    y = alt.Y('city:N', sort = city_order, title = 'City')
    ).properties(
    width = 500,
    height = 300,
    title = 'Price Distribution by City'
)
chart.save("charts/chart1.html")

Visualization 2

In [22]:
city_agg = df.groupby('city').agg(
    avg_price = ('price_total', 'mean'),
    listing_count = ('price_total', 'size'),
    salary = ('Monthly_Average_Net_salary', 'first')
).reset_index()

city_agg['affordability_ratio'] = (city_agg['avg_price'] * 30) / city_agg['salary']

plot = alt.Chart(city_agg).mark_circle(size = 200).encode(
    x = alt.X('salary:Q', title = 'Monthly Average Net Salary (€)'),
    y = alt.Y('avg_price:Q', title = 'Average Nightly Price (€)'),
    color = alt.Color('affordability_ratio:Q',
                      title = 'Affordability Ratio',
                      scale = alt.Scale(scheme = 'redyellowgreen', reverse = True)),
    tooltip = [
        alt.Tooltip('city:N'),
        alt.Tooltip('salary:Q', format = ',.0f'),
        alt.Tooltip('avg_price:Q', format = '.2f'),
        alt.Tooltip('affordability_ratio:Q', format = '.2f'),
        alt.Tooltip('listing_count:Q')
    ]
)

labels = alt.Chart(city_agg).mark_text(dy = -15).encode(
    x = alt.X('salary:Q'),
    y = alt.Y('avg_price:Q'),
    text = 'city:N'
)

chart = (plot + labels).properties(
    width = 550,
    height = 400,
    title = 'Affordability Gap: Nightly Price vs Local Salary by City'
).interactive()

chart.save("charts/chart2.html")

Oluwadamilare Viz

Visualization 3 (Done with D3 JS - see chart3.js)

In [23]:
brush = alt.selection_interval(encodings=['x'])

base = alt.Chart(df).encode(
    x=alt.X(
        'distance_city_center:Q',
        bin=alt.Bin(maxbins=30),
        title='Distance to City Center'
    ),
    y=alt.Y(
        'mean(price_total):Q',
        title='Average Price'
    ),
    color='room_type:N'
)

base = alt.Chart(df).encode(
    x=alt.X(
        'distance_city_center:Q',
        bin=alt.Bin(maxbins=30),
        title='Distance to City Center'
    ),
    y=alt.Y(
        'mean(price_total):Q',
        title='Average Price'
    ),
    color='room_type:N'
)

chart = base.mark_area(opacity=0.6).add_params(
    brush
).facet(
    column='room_type:N'
)

avg_price = alt.Chart(df).transform_filter(
    brush
).mark_text(size=16).encode(
    text=alt.Text('mean(price_total):Q', format='.2f')
).properties(
    title='Average Price (Selected Range)'
)

total = df.shape[0]

percent = alt.Chart(df).transform_filter(
    brush
).transform_aggregate(
    count_selected='count()'
).transform_calculate(
    pct=f'datum.count_selected / {total}'
).mark_text(size=16).encode(
    text=alt.Text('pct:Q', format='.1%')
).properties(
    title='% of Listings in Selection'
)

stats = alt.hconcat(avg_price, percent)

final_chart = alt.vconcat(
    chart,
    stats
)

final_chart
final_chart.save("charts/chart3.html")

Visualization 4

In [24]:
import numpy as np
import altair as alt

distance_order = ['<0.5km', '0.5-1km', '1-2km', '2-5km', '5km+']
room_order = ['Entire home/apt', 'Private room', 'Shared room']

df['distance_bin'] = pd.Categorical(
    pd.cut(
        df['distance_metro'],
        bins=[0, 0.5, 1, 2, 5, np.inf],
        labels=distance_order
    ),
    categories=distance_order,
    ordered=True
)

df['column_group'] = df['distance_bin'].astype(str) + ' | ' + df['room_type']
column_order = [f'{d} | {r}' for d in distance_order for r in room_order]

selection = alt.selection_point(fields=['column_group'], toggle=True)

heatmap = (
    alt.Chart(df)
    .mark_rect()
    .encode(
        y=alt.Y('city:N', title='City'),
        x=alt.X(
            'column_group:N',
            title='Distance Bin | Room Type',
            sort=column_order,
            axis=alt.Axis(labelAngle=-45, labelFontSize=9)
        ),
        color=alt.Color(
            'mean(price_total):Q',
            title='Avg Price (€)',
            scale=alt.Scale(scheme='yelloworangered')
        ),
        tooltip=[
            'city:N',
            'distance_bin:N',
            'room_type:N',
            alt.Tooltip('mean(price_total):Q', title='Avg Price', format='.0f')
        ],
        opacity=alt.condition(selection, alt.value(1), alt.value(0.3))
    )
    .add_params(selection)
    .properties(width=600, height=400, title='Average Price by City, Metro Distance, and Room Type')
)

filtered = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x=alt.X('city:N', title='City', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('mean(price_total):Q', title='Average Price (€)'),
        color=alt.Color('city:N', legend=None)
    )
    .transform_filter(selection)
    .properties(width=600, title='Average Price by City for Selected Distance Band and Room Type')
)

chart = heatmap & filtered
chart.save('charts/chart4.html')

Quinn Viz

Visualization 5

In [25]:
import altair as alt
agg = df.groupby(["city", "day_type"]).agg(
    mean_price=("price_total", "mean"),
    mean_sat=("guest_satisfaction_score", "mean")
).reset_index()

toggle = alt.param(
    name="view",
    value="price_view",
    bind=alt.binding_radio(
        options=["price_view", "satisfaction_view"],
        labels=["Price ($)", "Satisfaction score"],
        name="View: "
    )
)

color = alt.Color("day_type:N", legend=alt.Legend(title="Day type"))

def bar(y_field, y_title, condition_value, zero=True):
    return (
        alt.Chart(agg)
        .mark_bar()
        .encode(
            x=alt.X("city:N", title=None),
            xOffset="day_type:N",
            y=alt.Y(f"{y_field}:Q", title=y_title, scale=alt.Scale(zero=zero)),
            color=color,
            tooltip=["city", "day_type", alt.Tooltip(f"{y_field}:Q", format=".1f")],
            opacity=alt.condition(f"view == '{condition_value}'", alt.value(1), alt.value(0))
        )
    )

chart = (
    alt.layer(
        bar("mean_price", "Mean price ($)", "price_view"),
        bar("mean_sat", "Mean satisfaction score", "satisfaction_view", zero=False),
    )
    .add_params(toggle)
    .properties(width=600, height=350, title="Weekend Premium vs. Guest Satisfaction")
    .resolve_scale(y="independent")
)

chart.show()
chart.save("charts/chart5.html")

alt.LayerChart(...)